In [1]:
import re
import time
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
from google.colab import files

In [2]:
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

In [3]:
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Uploaded: []


In [4]:
REVIEWED_CSV = "step3_flagged_files_strict_review_completed.csv"
EVENTS_CSV = "events_clean.csv"

reviewed = pd.read_csv(REVIEWED_CSV)
events = pd.read_csv(EVENTS_CSV)

reviewed["manual_label"] = reviewed["manual_label"].astype(str).str.strip().str.lower()

repair_targets = reviewed.loc[
    reviewed["manual_label"].isin(["partial", "incorrect"])
].copy()

# standardize keys
repair_targets["filing_date"] = pd.to_datetime(
    repair_targets["filing_date"], dayfirst=True, errors="coerce"
).dt.strftime("%Y-%m-%d")

events["filing_date"] = pd.to_datetime(
    events["filing_date"], errors="coerce"
).dt.strftime("%Y-%m-%d")

repair_targets["ticker"] = repair_targets["ticker"].astype(str).str.strip().str.upper()
events["ticker"] = events["ticker"].astype(str).str.strip().str.upper()

repair_targets["filing_type"] = repair_targets["filing_type"].astype(str).str.strip().str.upper()
events["filing_type"] = events["filing_type"].astype(str).str.strip().str.upper()

repair_targets["accession_number"] = repair_targets["accession_number"].astype(str).str.strip()
events["accession_number"] = events["accession_number"].astype(str).str.strip()

repair_df = repair_targets.merge(
    events[["ticker", "cik", "filing_date", "filing_type", "accession_number"]],
    on=["ticker", "filing_date", "filing_type", "accession_number"],
    how="left"
)

print("Rows needing stronger repair:", len(repair_df))
print("Missing cik:", repair_df["cik"].isna().sum())
display(repair_df.head())

Rows needing stronger repair: 123
Missing cik: 0


,ticker,filing_date,filing_type,accession_number,mda_filename,word_count_actual,digit_ratio_full,good_start_detected,suspicious_start_detected,contains_market_risk_start,...,needs_manual_review,review_reason,manual_label,notes,notes_final,strict_label,strict_notes,original_manual_label,original_notes,cik
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_20191031_10-K_000032019319000119.txt,1018,0.015798,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
1,AAPL,2020-10-30,10-K,0000320193-20-000096,AAPL_20201030_10-K_000032019320000096.txt,1018,0.015847,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
2,AAPL,2021-10-29,10-K,0000320193-21-000105,AAPL_20211029_10-K_000032019321000105.txt,1018,0.015847,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
3,AAPL,2022-10-28,10-K,0000320193-22-000108,AAPL_20221028_10-K_000032019322000108.txt,1017,0.015720,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
4,ABT,2019-02-22,10-K,0001047469-19-000624,ABT_20190222_10-K_000104746919000624.txt,731,0.036641,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,1800


In [5]:
BASE_DIR = Path("/content/step3_strong_repair")
OUT_DIR = BASE_DIR / "repaired_texts_strong"
BASE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPAIR_LOG_CSV = BASE_DIR / "step3_strong_repair_log.csv"

In [6]:
HEADERS = {
    "User-Agent": "Cardiff University Student abhishekjc23@gmail.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov"
}

In [7]:
def cik_nolead(cik):
    if pd.isna(cik):
        return None
    return str(int(float(cik)))

def acc_nodash(acc):
    return str(acc).replace("-", "")

def filing_base_url(cik, accession_number):
    return f"https://www.sec.gov/Archives/edgar/data/{cik_nolead(cik)}/{acc_nodash(accession_number)}"

def index_json_url(cik, accession_number):
    return filing_base_url(cik, accession_number) + "/index.json"

def get_index_json(cik, accession_number):
    url = index_json_url(cik, accession_number)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def safe_int(value, default=0):
    try:
        if value is None:
            return default
        value = str(value).strip()
        if value == "":
            return default
        return int(value)
    except Exception:
        return default

In [8]:
def choose_primary_doc(index_json, filing_type):
    try:
        items = index_json["directory"]["item"]
    except Exception:
        return None

    docs = []
    for x in items:
        name = str(x.get("name", ""))
        lower = name.lower()
        if lower.endswith((".htm", ".html", ".txt")):
            docs.append(x)

    if not docs:
        return None

    bad_words = [
        "ex-", "exhibit", "xbrl", "xml", "graphic", "image", "zip",
        "def14a", "8-k", "news", "press"
    ]

    filtered = []
    for d in docs:
        nm = str(d.get("name", "")).lower()
        if any(b in nm for b in bad_words):
            continue
        filtered.append(d)

    if not filtered:
        filtered = docs

    filing_pref = []
    for d in filtered:
        nm = str(d.get("name", "")).lower().replace("-", "").replace("_", "")
        ft = filing_type.lower().replace("-", "").replace("_", "")
        if ft in nm:
            filing_pref.append(d)
        elif filing_type == "10-K" and "10k" in nm:
            filing_pref.append(d)
        elif filing_type == "10-Q" and "10q" in nm:
            filing_pref.append(d)

    if filing_pref:
        filtered = filing_pref

    filtered = sorted(
        filtered,
        key=lambda x: safe_int(x.get("size", 0), default=0),
        reverse=True
    )

    return filtered[0]["name"] if filtered else None

In [9]:
def download_filing_doc(cik, accession_number, filename):
    url = filing_base_url(cik, accession_number) + f"/{filename}"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def html_to_text(html):
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "ix:header", "header", "footer"]):
        tag.decompose()

    text = soup.get_text("\n")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

In [10]:
def remove_table_like_paragraphs(text, threshold=0.30):
    paras = re.split(r"\n\s*\n", text)
    kept = []

    for p in paras:
        p = p.strip()
        if not p:
            continue

        digits = sum(ch.isdigit() for ch in p)
        ratio = digits / max(len(p), 1)

        # keep headings even if short
        if len(p) < 200:
            kept.append(p)
            continue

        if ratio > threshold:
            continue

        kept.append(p)

    return "\n\n".join(kept)

In [11]:
def normalize_lines(text):
    lines = text.splitlines()
    cleaned = []
    for line in lines:
        line = line.replace("\xa0", " ")
        line = re.sub(r"[ \t]+", " ", line).strip()
        if line:
            cleaned.append(line)
    return cleaned

def is_toc_like(line):
    l = line.lower()
    return (
        ("table of contents" in l) or
        ("................" in l) or
        (re.search(r"item\s+\d+[a-z]?\s+.+\s+\d+$", l) is not None)
    )

def strong_start_patterns(filing_type):
    if filing_type == "10-K":
        return [
            re.compile(r"^item\s*7[\.\-:\s]+management['’` ]s discussion and analysis of financial condition and results of operations", re.I),
            re.compile(r"^management['’` ]s discussion and analysis of financial condition and results of operations", re.I),
        ]
    else:  # 10-Q
        return [
            re.compile(r"^item\s*2[\.\-:\s]+management['’` ]s discussion and analysis of financial condition and results of operations", re.I),
            re.compile(r"^management['’` ]s discussion and analysis of financial condition and results of operations", re.I),
        ]

def strong_end_patterns(filing_type):
    if filing_type == "10-K":
        return [
            re.compile(r"^item\s*7a[\.\-:\s]+quantitative and qualitative disclosures about market risk", re.I),
            re.compile(r"^item\s*8[\.\-:\s]+financial statements", re.I),
        ]
    else:  # 10-Q
        return [
            re.compile(r"^item\s*3[\.\-:\s]+quantitative and qualitative disclosures about market risk", re.I),
            re.compile(r"^item\s*4[\.\-:\s]+controls and procedures", re.I),
        ]

In [12]:
def extract_mda_by_lines(text, filing_type):
    lines = normalize_lines(text)

    # build candidate heading indices
    start_idx = []
    end_idx = []

    s_pats = strong_start_patterns(filing_type)
    e_pats = strong_end_patterns(filing_type)

    for i, line in enumerate(lines):
        # skip TOC-like lines
        if is_toc_like(line):
            continue

        # reject obvious references
        low = line.lower()
        if "under the heading" in low or "see item" in low or "refer to item" in low:
            continue

        for pat in s_pats:
            if pat.search(line):
                start_idx.append(i)
                break

        for pat in e_pats:
            if pat.search(line):
                end_idx.append(i)
                break

    if not start_idx:
        return None, "NO_START"

    if not end_idx:
        return None, "NO_END"

    # choose first valid start-end pair with realistic span
    best_pair = None
    for s in start_idx:
        candidates = [e for e in end_idx if e > s]
        if not candidates:
            continue
        e = candidates[0]

        # ignore starts too early in document if followed by Part I / Item 1 business
        early_block = " ".join(lines[max(0, s-5):min(len(lines), s+5)]).lower()
        if "part i" in early_block and "item 1" in early_block and "business" in early_block:
            continue

        span_lines = e - s
        if 20 <= span_lines <= 5000:
            best_pair = (s, e)
            break

    if best_pair is None:
        return None, "NO_VALID_SPAN"

    s, e = best_pair
    mda_text = "\n".join(lines[s:e]).strip()

    wc = len(re.findall(r"\b[a-zA-Z]+\b", mda_text))
    if wc < 800:
        return None, "TOO_SHORT"

    # final guard: start must really look like heading
    first_300 = re.sub(r"\s+", " ", mda_text[:300]).lower()
    if filing_type == "10-K":
        if ("item 7" not in first_300) and ("management's discussion" not in first_300 and "management’s discussion" not in first_300):
            return None, "BAD_START"
    else:
        if ("item 2" not in first_300) and ("management's discussion" not in first_300 and "management’s discussion" not in first_300):
            return None, "BAD_START"

    return mda_text, "OK"

In [13]:
def repair_one_filing_strong(row, sleep_seconds=0.2):
    ticker = row["ticker"]
    cik = row["cik"]
    filing_date = row["filing_date"]
    filing_type = row["filing_type"]
    accession_number = row["accession_number"]

    out = {
        "ticker": ticker,
        "filing_date": filing_date,
        "filing_type": filing_type,
        "accession_number": accession_number,
        "original_manual_label": row["manual_label"],
        "repair_status": "",
        "primary_doc": "",
        "repaired_filename": ""
    }

    try:
        if pd.isna(cik):
            out["repair_status"] = "NO_CIK"
            return out

        idx = get_index_json(cik, accession_number)
        primary_doc = choose_primary_doc(idx, filing_type)

        if primary_doc is None:
            out["repair_status"] = "NO_PRIMARY_DOC"
            return out

        out["primary_doc"] = primary_doc

        html = download_filing_doc(cik, accession_number, primary_doc)
        text = html_to_text(html)
        text = remove_table_like_paragraphs(text, threshold=0.30)

        mda_text, status = extract_mda_by_lines(text, filing_type)

        if status != "OK":
            out["repair_status"] = status
            return out

        repaired_filename = f"{ticker}_{filing_date}_{filing_type}_{accession_number}_REPAIRED_STRONG.txt"
        repaired_path = OUT_DIR / repaired_filename

        with open(repaired_path, "w", encoding="utf-8") as f:
            f.write(mda_text)

        out["repair_status"] = "OK"
        out["repaired_filename"] = repaired_filename

        time.sleep(sleep_seconds)
        return out

    except Exception as e:
        out["repair_status"] = f"ERROR: {str(e)[:120]}"
        return out

In [14]:
strong_results = []

for i, row in repair_df.iterrows():
    result = repair_one_filing_strong(row)
    strong_results.append(result)

    if (i + 1) % 10 == 0:
        print(f"Processed {i+1}/{len(repair_df)}")

strong_log = pd.DataFrame(strong_results)
strong_log.to_csv(REPAIR_LOG_CSV, index=False)

print("Strong repair complete.")
display(strong_log["repair_status"].value_counts())
display(strong_log.head())

Processed 10/123
Processed 20/123
Processed 30/123
Processed 40/123
Processed 50/123
Processed 60/123
Processed 70/123
Processed 80/123
Processed 90/123
Processed 100/123
Processed 110/123
Processed 120/123
Strong repair complete.


,count
repair_status,
OK,59
NO_START,41
NO_END,19
NO_VALID_SPAN,4


,ticker,filing_date,filing_type,accession_number,original_manual_label,repair_status,primary_doc,repaired_filename
0,AAPL,2019-10-31,10-K,0000320193-19-000119,incorrect,NO_END,a10-k20199282019.htm,
1,AAPL,2020-10-30,10-K,0000320193-20-000096,incorrect,OK,aapl-20200926.htm,AAPL_2020-10-30_10-K_0000320193-20-000096_REPA...
2,AAPL,2021-10-29,10-K,0000320193-21-000105,incorrect,OK,aapl-20210925.htm,AAPL_2021-10-29_10-K_0000320193-21-000105_REPA...
3,AAPL,2022-10-28,10-K,0000320193-22-000108,incorrect,OK,aapl-20220924.htm,AAPL_2022-10-28_10-K_0000320193-22-000108_REPA...
4,ABT,2019-02-22,10-K,0001047469-19-000624,incorrect,OK,a2237733z10-k.htm,ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...


In [15]:
def preview_repaired_file_strong(filename, start_chars=2500, end_chars=1200):
    path = OUT_DIR / filename

    if not path.exists():
        print("File not found:", path)
        return

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    print("=" * 120)
    print("FILE:", filename)
    print("=" * 120)
    print("\n--- START PREVIEW ---\n")
    print(text[:start_chars])
    print("\n--- END PREVIEW ---\n")
    print(text[-end_chars:])
    print("\nWord count:", len(re.findall(r"\b[a-zA-Z]+\b", text)))
    print("=" * 120)

In [16]:
ok_strong = strong_log.loc[
    strong_log["repair_status"] == "OK", "repaired_filename"
].dropna().tolist()

print("Strongly repaired OK files:", len(ok_strong))
print(ok_strong[:20])

Strongly repaired OK files: 59
['AAPL_2020-10-30_10-K_0000320193-20-000096_REPAIRED_STRONG.txt', 'AAPL_2021-10-29_10-K_0000320193-21-000105_REPAIRED_STRONG.txt', 'AAPL_2022-10-28_10-K_0000320193-22-000108_REPAIRED_STRONG.txt', 'ABT_2019-02-22_10-K_0001047469-19-000624_REPAIRED_STRONG.txt', 'ABT_2020-02-21_10-K_0001104659-20-023904_REPAIRED_STRONG.txt', 'ABT_2021-02-19_10-K_0001104659-21-025751_REPAIRED_STRONG.txt', 'ABT_2022-02-18_10-K_0001104659-22-025141_REPAIRED_STRONG.txt', 'APH_2022-02-09_10-K_0001558370-22-000961_REPAIRED_STRONG.txt', 'APH_2023-02-08_10-K_0001558370-23-001036_REPAIRED_STRONG.txt', 'APH_2024-02-07_10-K_0001558370-24-000866_REPAIRED_STRONG.txt', 'BKNG_2019-05-09_10-Q_0001075531-19-000021_REPAIRED_STRONG.txt', 'BKNG_2019-08-07_10-Q_0001075531-19-000043_REPAIRED_STRONG.txt', 'BKNG_2019-11-07_10-Q_0001075531-19-000064_REPAIRED_STRONG.txt', 'CB_2021-02-25_10-K_0000896159-21-000003_REPAIRED_STRONG.txt', 'CB_2022-02-24_10-K_0000896159-22-000005_REPAIRED_STRONG.txt', 'CB_

In [17]:
import re
import pandas as pd

ok_strong_df = strong_log.loc[
    strong_log["repair_status"] == "OK",
    ["ticker", "filing_date", "filing_type", "accession_number", "repaired_filename"]
].dropna().copy()

print("OK strong repairs:", len(ok_strong_df))
display(ok_strong_df.head())

OK strong repairs: 59


,ticker,filing_date,filing_type,accession_number,repaired_filename
1,AAPL,2020-10-30,10-K,0000320193-20-000096,AAPL_2020-10-30_10-K_0000320193-20-000096_REPA...
2,AAPL,2021-10-29,10-K,0000320193-21-000105,AAPL_2021-10-29_10-K_0000320193-21-000105_REPA...
3,AAPL,2022-10-28,10-K,0000320193-22-000108,AAPL_2022-10-28_10-K_0000320193-22-000108_REPA...
4,ABT,2019-02-22,10-K,0001047469-19-000624,ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...
5,ABT,2020-02-21,10-K,0001104659-20-023904,ABT_2020-02-21_10-K_0001104659-20-023904_REPAI...


In [18]:
def read_text_safe(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def first_chunk(text, n=1500):
    return re.sub(r"\s+", " ", text[:n]).strip()

def word_count_alpha(text):
    return len(re.findall(r"\b[a-zA-Z]+\b", text))

In [19]:
def validate_repaired_start(text, filing_type):
    t = first_chunk(text, 1500).lower()
    t = t.replace("’", "'")

    if filing_type == "10-K":
        strong_heading = bool(re.search(
            r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            t
        ))
        weaker_heading = bool(re.search(
            r"management[' ]s discussion and analysis of financial condition and results of operations",
            t
        ))
    else:  # 10-Q
        strong_heading = bool(re.search(
            r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            t
        ))
        weaker_heading = bool(re.search(
            r"management[' ]s discussion and analysis of financial condition and results of operations",
            t
        ))

    bad_reference = (
        "under the heading" in t or
        "see part ii, item 7" in t or
        "see item 2" in t or
        "refer to item" in t or
        "part i item 1" in t or
        "company background" in t or
        "business" in t[:400]
    )

    return {
        "strong_heading_match": strong_heading,
        "weaker_heading_match": weaker_heading,
        "bad_reference_start": bad_reference,
        "first_300_chars": text[:300].replace("\n", " ")
    }

In [20]:
records = []

for _, row in ok_strong_df.iterrows():
    path = OUT_DIR / row["repaired_filename"]
    text = read_text_safe(path)
    checks = validate_repaired_start(text, row["filing_type"])

    records.append({
        "ticker": row["ticker"],
        "filing_date": row["filing_date"],
        "filing_type": row["filing_type"],
        "accession_number": row["accession_number"],
        "repaired_filename": row["repaired_filename"],
        "word_count": word_count_alpha(text),
        **checks
    })

ok_validation_df = pd.DataFrame(records)
display(ok_validation_df.head())

,ticker,filing_date,filing_type,accession_number,repaired_filename,word_count,strong_heading_match,weaker_heading_match,bad_reference_start,first_300_chars
0,AAPL,2020-10-30,10-K,0000320193-20-000096,AAPL_2020-10-30_10-K_0000320193-20-000096_REPA...,16716,False,True,True,Management’s Discussion and Analysis of Financ...
1,AAPL,2021-10-29,10-K,0000320193-21-000105,AAPL_2021-10-29_10-K_0000320193-21-000105_REPA...,16202,False,True,True,Management’s Discussion and Analysis of Financ...
2,AAPL,2022-10-28,10-K,0000320193-22-000108,AAPL_2022-10-28_10-K_0000320193-22-000108_REPA...,16239,False,True,True,Management’s Discussion and Analysis of Financ...
3,ABT,2019-02-22,10-K,0001047469-19-000624,ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...,13165,True,True,False,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...
4,ABT,2020-02-21,10-K,0001104659-20-023904,ABT_2020-02-21_10-K_0001104659-20-023904_REPAI...,12692,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...


In [21]:
ok_validation_df["needs_manual_review"] = (
    (~ok_validation_df["strong_heading_match"]) |
    (ok_validation_df["bad_reference_start"])
)

likely_good_ok = ok_validation_df.loc[~ok_validation_df["needs_manual_review"]].copy()
needs_review_ok = ok_validation_df.loc[ok_validation_df["needs_manual_review"]].copy()

print("Likely good repaired files:", len(likely_good_ok))
print("Repaired files needing manual review:", len(needs_review_ok))

Likely good repaired files: 30
Repaired files needing manual review: 29


In [22]:
needs_review_ok["manual_label"] = ""
needs_review_ok["notes"] = ""

likely_good_ok_sample = likely_good_ok.sample(min(10, len(likely_good_ok)), random_state=42).copy()
likely_good_ok_sample["manual_label"] = ""
likely_good_ok_sample["notes"] = ""

needs_review_ok.to_csv("strong_repaired_files_needing_review.csv", index=False)
likely_good_ok_sample.to_csv("strong_repaired_files_likely_good_sample.csv", index=False)

print("Saved strong_repaired_files_needing_review.csv")
print("Saved strong_repaired_files_likely_good_sample.csv")

Saved strong_repaired_files_needing_review.csv
Saved strong_repaired_files_likely_good_sample.csv


In [23]:
def preview_repaired_file_strong(filename, start_chars=2500, end_chars=1200):
    path = OUT_DIR / filename

    if not path.exists():
        print("File not found:", path)
        return

    text = read_text_safe(path)

    print("=" * 120)
    print("FILE:", filename)
    print("=" * 120)
    print("\n--- START PREVIEW ---\n")
    print(text[:start_chars])
    print("\n--- END PREVIEW ---\n")
    print(text[-end_chars:])
    print("\nWord count:", word_count_alpha(text))
    print("=" * 120)

In [24]:
likely_good_ok["manual_label"] = ""
likely_good_ok["notes"] = ""

likely_good_ok.to_csv("strong_repaired_files_likely_good_all.csv", index=False)

print("Saved full likely-good repaired file list:", len(likely_good_ok))

Saved full likely-good repaired file list: 30


In [25]:
import zipfile

STRONG_ZIP = str(BASE_DIR / "step3_repaired_texts_strong.zip")

with zipfile.ZipFile(STRONG_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for txt_file in OUT_DIR.glob("*.txt"):
        zipf.write(txt_file, arcname=txt_file.name)

print("Saved:", STRONG_ZIP)

Saved: /content/step3_strong_repair/step3_repaired_texts_strong.zip


In [26]:
accepted_repairs = pd.read_csv("strong_repaired_files_likely_good_all.csv")

accepted_repairs["manual_label"] = accepted_repairs["manual_label"].astype(str).str.strip().str.lower()

accepted_repairs = accepted_repairs.loc[
    accepted_repairs["manual_label"] == "correct"
].copy()

print("Accepted repaired rows:", len(accepted_repairs))
display(accepted_repairs.head())

Accepted repaired rows: 0


,ticker,filing_date,filing_type,accession_number,repaired_filename,word_count,strong_heading_match,weaker_heading_match,bad_reference_start,first_300_chars,needs_manual_review,manual_label,notes


In [27]:
accepted_repairs = pd.read_csv("strong_repaired_files_likely_good_all.csv")

accepted_repairs["manual_label"] = accepted_repairs["manual_label"].fillna("").astype(str).str.strip().str.lower()
accepted_repairs["notes"] = accepted_repairs["notes"].fillna("").astype(str)

correct_keys = {
    ("ABT", "2019-02-22", "10-K"),
    ("ABT", "2020-02-21", "10-K"),
    ("ABT", "2021-02-19", "10-K"),
    ("ABT", "2022-02-18", "10-K"),

    ("BKNG", "2019-05-09", "10-Q"),
    ("BKNG", "2019-08-07", "10-Q"),
    ("BKNG", "2019-11-07", "10-Q"),

    ("CB", "2021-02-25", "10-K"),
    ("CB", "2022-02-24", "10-K"),
    ("CB", "2024-02-23", "10-K"),

    ("JCI", "2019-08-01", "10-Q"),
    ("JCI", "2020-01-31", "10-Q"),
    ("JCI", "2020-05-01", "10-Q"),

    ("KKR", "2023-05-10", "10-Q"),
    ("KKR", "2023-08-08", "10-Q"),
    ("KKR", "2023-11-09", "10-Q"),
    ("KKR", "2024-02-29", "10-K"),
    ("KKR", "2024-05-09", "10-Q"),
    ("KKR", "2024-08-09", "10-Q"),
    ("KKR", "2024-11-05", "10-Q"),

    ("MO", "2020-04-30", "10-Q"),
    ("MO", "2020-10-30", "10-Q"),
}

accepted_repairs["ticker"] = accepted_repairs["ticker"].astype(str).str.strip().str.upper()
accepted_repairs["filing_date"] = pd.to_datetime(accepted_repairs["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")
accepted_repairs["filing_type"] = accepted_repairs["filing_type"].astype(str).str.strip().str.upper()

accepted_repairs.loc[
    accepted_repairs.apply(
        lambda r: (r["ticker"], r["filing_date"], r["filing_type"]) in correct_keys,
        axis=1
    ),
    ["manual_label", "notes"]
] = ["correct", "accepted after strict repaired-file review"]

print(accepted_repairs["manual_label"].value_counts(dropna=False))
display(accepted_repairs.loc[accepted_repairs["manual_label"] == "correct"].head(30))

manual_label
correct    22
            8
Name: count, dtype: int64


,ticker,filing_date,filing_type,accession_number,repaired_filename,word_count,strong_heading_match,weaker_heading_match,bad_reference_start,first_300_chars,needs_manual_review,manual_label,notes
0,ABT,2019-02-22,10-K,0001047469-19-000624,ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...,13165,True,True,False,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
1,ABT,2020-02-21,10-K,0001104659-20-023904,ABT_2020-02-21_10-K_0001104659-20-023904_REPAI...,12692,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
2,ABT,2021-02-19,10-K,0001104659-21-025751,ABT_2021-02-19_10-K_0001104659-21-025751_REPAI...,10762,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
3,ABT,2022-02-18,10-K,0001104659-22-025141,ABT_2022-02-18_10-K_0001104659-22-025141_REPAI...,10747,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
4,BKNG,2019-05-09,10-Q,0001075531-19-000021,BKNG_2019-05-09_10-Q_0001075531-19-000021_REPA...,9818,True,True,False,Item 2. Management's Discussion and Analysis o...,False,correct,accepted after strict repaired-file review
5,BKNG,2019-08-07,10-Q,0001075531-19-000043,BKNG_2019-08-07_10-Q_0001075531-19-000043_REPA...,10476,True,True,False,Item 2. Management's Discussion and Analysis o...,False,correct,accepted after strict repaired-file review
6,BKNG,2019-11-07,10-Q,0001075531-19-000064,BKNG_2019-11-07_10-Q_0001075531-19-000064_REPA...,10675,True,True,False,Item 2. Management's Discussion and Analysis o...,False,correct,accepted after strict repaired-file review
7,CB,2021-02-25,10-K,0000896159-21-000003,CB_2021-02-25_10-K_0000896159-21-000003_REPAIR...,25767,True,True,False,ITEM 7. Management's Discussion and Analysis o...,False,correct,accepted after strict repaired-file review
8,CB,2022-02-24,10-K,0000896159-22-000005,CB_2022-02-24_10-K_0000896159-22-000005_REPAIR...,25827,True,True,False,ITEM 7. Management's Discussion and Analysis o...,False,correct,accepted after strict repaired-file review
9,CB,2024-02-23,10-K,0000896159-24-000003,CB_2024-02-23_10-K_0000896159-24-000003_REPAIR...,26752,True,True,False,ITEM 7. Management's Discussion and Analysis o...,False,correct,accepted after strict repaired-file review


In [28]:
accepted_repairs.to_csv("strong_repaired_files_likely_good_all_reviewed.csv", index=False)
print("Saved reviewed accepted repairs file.")

Saved reviewed accepted repairs file.


In [29]:
accepted_repairs = pd.read_csv("strong_repaired_files_likely_good_all_reviewed.csv")

accepted_repairs["manual_label"] = accepted_repairs["manual_label"].astype(str).str.strip().str.lower()

accepted_repairs = accepted_repairs.loc[
    accepted_repairs["manual_label"] == "correct"
].copy()

print("Accepted repaired rows:", len(accepted_repairs))
display(accepted_repairs.head())

Accepted repaired rows: 22


,ticker,filing_date,filing_type,accession_number,repaired_filename,word_count,strong_heading_match,weaker_heading_match,bad_reference_start,first_300_chars,needs_manual_review,manual_label,notes
0,ABT,2019-02-22,10-K,0001047469-19-000624,ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...,13165,True,True,False,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
1,ABT,2020-02-21,10-K,0001104659-20-023904,ABT_2020-02-21_10-K_0001104659-20-023904_REPAI...,12692,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
2,ABT,2021-02-19,10-K,0001104659-21-025751,ABT_2021-02-19_10-K_0001104659-21-025751_REPAI...,10762,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
3,ABT,2022-02-18,10-K,0001104659-22-025141,ABT_2022-02-18_10-K_0001104659-22-025141_REPAI...,10747,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
4,BKNG,2019-05-09,10-Q,0001075531-19-000021,BKNG_2019-05-09_10-Q_0001075531-19-000021_REPA...,9818,True,True,False,Item 2. Management's Discussion and Analysis o...,False,correct,accepted after strict repaired-file review


In [31]:
accepted_repairs["ticker"] = accepted_repairs["ticker"].astype(str).str.strip().str.upper()
accepted_repairs["filing_date"] = pd.to_datetime(accepted_repairs["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")
accepted_repairs["filing_type"] = accepted_repairs["filing_type"].astype(str).str.strip().str.upper()
accepted_repairs["accession_number"] = accepted_repairs["accession_number"].astype(str).str.strip()

accepted_keys = set(
    zip(
        accepted_repairs["ticker"],
        accepted_repairs["filing_date"],
        accepted_repairs["filing_type"],
        accepted_repairs["accession_number"]
    )
)

print("Accepted repaired keys:", len(accepted_keys))

Accepted repaired keys: 22


In [32]:
reviewed["ticker"] = reviewed["ticker"].astype(str).str.strip().str.upper()
reviewed["filing_date"] = pd.to_datetime(reviewed["filing_date"], dayfirst=True, errors="coerce").dt.strftime("%Y-%m-%d")
reviewed["filing_type"] = reviewed["filing_type"].astype(str).str.strip().str.upper()
reviewed["accession_number"] = reviewed["accession_number"].astype(str).str.strip()
reviewed["manual_label"] = reviewed["manual_label"].astype(str).str.strip().str.lower()

In [33]:
accepted_repairs = accepted_repairs.loc[
    accepted_repairs["manual_label"] == "correct"
].copy()

accepted_keys = set(
    zip(
        accepted_repairs["ticker"],
        accepted_repairs["filing_date"],
        accepted_repairs["filing_type"],
        accepted_repairs["accession_number"]
    )
)

print("Accepted repaired rows:", len(accepted_repairs))
display(accepted_repairs[["ticker","filing_date","filing_type","accession_number","repaired_filename"]].head(30))

Accepted repaired rows: 22


,ticker,filing_date,filing_type,accession_number,repaired_filename
0,ABT,2019-02-22,10-K,0001047469-19-000624,ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...
1,ABT,2020-02-21,10-K,0001104659-20-023904,ABT_2020-02-21_10-K_0001104659-20-023904_REPAI...
2,ABT,2021-02-19,10-K,0001104659-21-025751,ABT_2021-02-19_10-K_0001104659-21-025751_REPAI...
3,ABT,2022-02-18,10-K,0001104659-22-025141,ABT_2022-02-18_10-K_0001104659-22-025141_REPAI...
4,BKNG,2019-05-09,10-Q,0001075531-19-000021,BKNG_2019-05-09_10-Q_0001075531-19-000021_REPA...
5,BKNG,2019-08-07,10-Q,0001075531-19-000043,BKNG_2019-08-07_10-Q_0001075531-19-000043_REPA...
6,BKNG,2019-11-07,10-Q,0001075531-19-000064,BKNG_2019-11-07_10-Q_0001075531-19-000064_REPA...
7,CB,2021-02-25,10-K,0000896159-21-000003,CB_2021-02-25_10-K_0000896159-21-000003_REPAIR...
8,CB,2022-02-24,10-K,0000896159-22-000005,CB_2022-02-24_10-K_0000896159-22-000005_REPAIR...
9,CB,2024-02-23,10-K,0000896159-24-000003,CB_2024-02-23_10-K_0000896159-24-000003_REPAIR...


In [34]:
remaining_targets = reviewed.loc[
    reviewed["manual_label"].isin(["partial", "incorrect"])
].copy()

remaining_targets = remaining_targets.loc[
    ~remaining_targets.apply(
        lambda r: (r["ticker"], r["filing_date"], r["filing_type"], r["accession_number"]) in accepted_keys,
        axis=1
    )
].copy()

print("Remaining unresolved files:", len(remaining_targets))
display(remaining_targets[["ticker","filing_date","filing_type","accession_number","manual_label"]].head())

Remaining unresolved files: 101


,ticker,filing_date,filing_type,accession_number,manual_label
0,AAPL,2019-10-31,10-K,0000320193-19-000119,incorrect
1,AAPL,2020-10-30,10-K,0000320193-20-000096,incorrect
2,AAPL,2021-10-29,10-K,0000320193-21-000105,incorrect
3,AAPL,2022-10-28,10-K,0000320193-22-000108,incorrect
8,ABT,2023-02-17,10-K,0001628280-23-004026,incorrect


In [38]:
import os
import zipfile
import shutil
import pandas as pd
from pathlib import Path
from google.colab import files

In [39]:
accepted_repairs = pd.read_csv("strong_repaired_files_likely_good_all_reviewed.csv")
print("Rows loaded:", len(accepted_repairs))
display(accepted_repairs.head())

Rows loaded: 30


,ticker,filing_date,filing_type,accession_number,repaired_filename,word_count,strong_heading_match,weaker_heading_match,bad_reference_start,first_300_chars,needs_manual_review,manual_label,notes
0,ABT,2019-02-22,10-K,0001047469-19-000624,ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...,13165,True,True,False,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
1,ABT,2020-02-21,10-K,0001104659-20-023904,ABT_2020-02-21_10-K_0001104659-20-023904_REPAI...,12692,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
2,ABT,2021-02-19,10-K,0001104659-21-025751,ABT_2021-02-19_10-K_0001104659-21-025751_REPAI...,10762,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
3,ABT,2022-02-18,10-K,0001104659-22-025141,ABT_2022-02-18_10-K_0001104659-22-025141_REPAI...,10747,True,True,False,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...,False,correct,accepted after strict repaired-file review
4,BKNG,2019-05-09,10-Q,0001075531-19-000021,BKNG_2019-05-09_10-Q_0001075531-19-000021_REPA...,9818,True,True,False,Item 2. Management's Discussion and Analysis o...,False,correct,accepted after strict repaired-file review


In [40]:
accepted_repairs["ticker"] = accepted_repairs["ticker"].astype(str).str.strip().str.upper()
accepted_repairs["filing_type"] = accepted_repairs["filing_type"].astype(str).str.strip().str.upper()
accepted_repairs["accession_number"] = accepted_repairs["accession_number"].astype(str).str.strip()
accepted_repairs["filing_date"] = pd.to_datetime(
    accepted_repairs["filing_date"], errors="coerce"
).dt.strftime("%Y-%m-%d")
accepted_repairs["manual_label"] = accepted_repairs["manual_label"].astype(str).str.strip().str.lower()

accepted_repairs = accepted_repairs.loc[
    accepted_repairs["manual_label"] == "correct"
].copy()

print("Accepted repaired rows:", len(accepted_repairs))
display(accepted_repairs[["ticker","filing_date","filing_type","accession_number","repaired_filename"]].head(30))

Accepted repaired rows: 22


,ticker,filing_date,filing_type,accession_number,repaired_filename
0,ABT,2019-02-22,10-K,0001047469-19-000624,ABT_2019-02-22_10-K_0001047469-19-000624_REPAI...
1,ABT,2020-02-21,10-K,0001104659-20-023904,ABT_2020-02-21_10-K_0001104659-20-023904_REPAI...
2,ABT,2021-02-19,10-K,0001104659-21-025751,ABT_2021-02-19_10-K_0001104659-21-025751_REPAI...
3,ABT,2022-02-18,10-K,0001104659-22-025141,ABT_2022-02-18_10-K_0001104659-22-025141_REPAI...
4,BKNG,2019-05-09,10-Q,0001075531-19-000021,BKNG_2019-05-09_10-Q_0001075531-19-000021_REPA...
5,BKNG,2019-08-07,10-Q,0001075531-19-000043,BKNG_2019-08-07_10-Q_0001075531-19-000043_REPA...
6,BKNG,2019-11-07,10-Q,0001075531-19-000064,BKNG_2019-11-07_10-Q_0001075531-19-000064_REPA...
7,CB,2021-02-25,10-K,0000896159-21-000003,CB_2021-02-25_10-K_0000896159-21-000003_REPAIR...
8,CB,2022-02-24,10-K,0000896159-22-000005,CB_2022-02-24_10-K_0000896159-22-000005_REPAIR...
9,CB,2024-02-23,10-K,0000896159-24-000003,CB_2024-02-23_10-K_0000896159-24-000003_REPAIR...


In [41]:
STRONG_REPAIRED_DIR = Path("/content/step3_strong_repair/repaired_texts_strong")
print("Folder exists:", STRONG_REPAIRED_DIR.exists())
print("Txt files in folder:", len(list(STRONG_REPAIRED_DIR.glob("*.txt"))))

Folder exists: True
Txt files in folder: 59


In [42]:
DOWNLOAD_DIR = Path("/content/accepted_22_repaired_txt")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# clear old files if any
for old_file in DOWNLOAD_DIR.glob("*.txt"):
    old_file.unlink()

copied = 0
missing = []

for _, row in accepted_repairs.iterrows():
    fname = row["repaired_filename"]
    src = STRONG_REPAIRED_DIR / fname
    dst = DOWNLOAD_DIR / fname

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(fname)

print("Copied files:", copied)
print("Missing files:", len(missing))
if missing:
    print("Missing list:")
    for x in missing:
        print(x)

Copied files: 22
Missing files: 0


In [43]:
ACCEPTED_22_CSV = "/content/accepted_22_repaired_files.csv"
accepted_repairs.to_csv(ACCEPTED_22_CSV, index=False)
print("Saved:", ACCEPTED_22_CSV)

Saved: /content/accepted_22_repaired_files.csv


In [44]:
ACCEPTED_22_ZIP = "/content/accepted_22_repaired_txt.zip"

with zipfile.ZipFile(ACCEPTED_22_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for txt_file in DOWNLOAD_DIR.glob("*.txt"):
        zipf.write(txt_file, arcname=txt_file.name)

print("Saved zip:", ACCEPTED_22_ZIP)

Saved zip: /content/accepted_22_repaired_txt.zip


In [45]:
reviewed = pd.read_csv("step3_flagged_files_strict_review_completed.csv")

reviewed["ticker"] = reviewed["ticker"].astype(str).str.strip().str.upper()
reviewed["filing_type"] = reviewed["filing_type"].astype(str).str.strip().str.upper()
reviewed["accession_number"] = reviewed["accession_number"].astype(str).str.strip()
reviewed["filing_date"] = pd.to_datetime(
    reviewed["filing_date"], dayfirst=True, errors="coerce"
).dt.strftime("%Y-%m-%d")
reviewed["manual_label"] = reviewed["manual_label"].astype(str).str.strip().str.lower()

accepted_keys = set(
    zip(
        accepted_repairs["ticker"],
        accepted_repairs["filing_date"],
        accepted_repairs["filing_type"],
        accepted_repairs["accession_number"]
    )
)

remaining_101 = reviewed.loc[
    reviewed["manual_label"].isin(["partial", "incorrect"])
].copy()

remaining_101 = remaining_101.loc[
    ~remaining_101.apply(
        lambda r: (r["ticker"], r["filing_date"], r["filing_type"], r["accession_number"]) in accepted_keys,
        axis=1
    )
].copy()

print("Remaining unresolved files:", len(remaining_101))
display(remaining_101.head())

Remaining unresolved files: 101


,ticker,filing_date,filing_type,accession_number,mda_filename,word_count_actual,digit_ratio_full,good_start_detected,suspicious_start_detected,contains_market_risk_start,...,first_400_chars,needs_manual_review,review_reason,manual_label,notes,notes_final,strict_label,strict_notes,original_manual_label,original_notes
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_20191031_10-K_000032019319000119.txt,1018,0.015798,False,True,True,...,A. Quantitative and Qualitative Disclosures Ab...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
1,AAPL,2020-10-30,10-K,0000320193-20-000096,AAPL_20201030_10-K_000032019320000096.txt,1018,0.015847,False,True,True,...,A. Quantitative and Qualitative Disclosures...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
2,AAPL,2021-10-29,10-K,0000320193-21-000105,AAPL_20211029_10-K_000032019321000105.txt,1018,0.015847,False,True,True,...,A. Quantitative and Qualitative Disclosures...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
3,AAPL,2022-10-28,10-K,0000320193-22-000108,AAPL_20221028_10-K_000032019322000108.txt,1017,0.015720,False,True,True,...,A. Quantitative and Qualitative Disclosures...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
8,ABT,2023-02-17,10-K,0001628280-23-004026,ABT_20230217_10-K_000162828023004026.txt,723,0.060734,False,True,True,...,A. QUANTITATIVE AND QUALITATIVE DISCLOSURES AB...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN


In [46]:
REMAINING_101_CSV = "/content/remaining_101_files.csv"
remaining_101.to_csv(REMAINING_101_CSV, index=False)
print("Saved:", REMAINING_101_CSV)
files.download(REMAINING_101_CSV)

Saved: /content/remaining_101_files.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>